# سیستم RAG برای استخراج خواص فیزیکی مواد از مقالات

**قبل از شروع:** منوی *Runtime → Change runtime type* → **T4 GPU**

سلول‌های ۱ تا ۵ را **به ترتیب** اجرا کنید (دکمه‌ی ▶). بعد از سلول ۵ برنامه باز می‌شود و همه‌ی کارها
(آپلود مقاله، پرسش و پاسخ، ساخت فایل برای چت، استخراج جدول خواص) داخل برنامه انجام می‌شود.

همه‌چیز (مقاله‌ها، پایگاه داده، نتیجه‌ها) در **Google Drive** در پوشه‌ی `matrag_data` ذخیره می‌شود و با بستن کولب پاک نمی‌شود.
دفعه‌ی بعد فقط کافی است دوباره سلول‌های ۱ تا ۵ را اجرا کنید.

In [ ]:
#@title ۱. بررسی کارت گرافیک
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ کارت گرافیک فعال نیست: Runtime → Change runtime type → T4 GPU"

In [ ]:
#@title ۲. اتصال به Google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/matrag_data'
os.makedirs(f'{WORK}/logs', exist_ok=True)
os.environ['MATRAG_DATA_DIR'] = f'{WORK}/data'
# The vector database runs on Colab's local disk (SQLite is unreliable on Drive);
# the app copies it back to Drive after every change.
os.environ['MATRAG_STORAGE_DIR'] = '/content/storage'
os.environ['MATRAG_STORAGE_BACKUP_DIR'] = f'{WORK}/storage'
!mkdir -p "{WORK}/storage" /content/storage && cp -rT "{WORK}/storage" /content/storage
print('✅ Data folder:', WORK)

In [ ]:
#@title ۳. نصب برنامه از GitHub
BRANCH = "claude/festive-bardeen-nnw123"  #@param {type:"string"}
# Leave the project folder before deleting it (re-running this cell used to fail
# with "Unable to read current working directory").
%cd /content
!rm -rf /content/RAG && git clone -q --depth 1 -b {BRANCH} https://github.com/aminsadidi/RAG.git /content/RAG
%cd /content/RAG
!pip install -q -e . 2>&1 | grep -iE "^error" || true
# An editable install is only seen by Python after a restart; adding the source
# folder to the path makes it importable in this session right away.
import sys
if '/content/RAG/src' not in sys.path:
    sys.path.insert(0, '/content/RAG/src')
!git log --oneline -1
!matrag --help > /dev/null && echo '✅ Installed (red "dependency conflict" warnings from pip can be ignored)' || echo '❌ Installation failed: run this cell again and send the output'


In [ ]:
#@title ۴. راه‌اندازی مدل زبانی
#@markdown **ollama**: مدل متن‌باز روی کارت گرافیک کولب (رایگان، بدون سهمیه). بار اول دانلود مدل چند دقیقه طول می‌کشد.
#@markdown **gemini**: کلید API را در بخش Secrets کولب (آیکون 🔑 سمت چپ) با نام `MATRAG_GOOGLE_API_KEY` ذخیره کنید.
LLM_PROVIDER = "ollama"  #@param ["ollama", "gemini"]
OLLAMA_MODEL = "qwen3:14b"  #@param ["qwen3:14b", "qwen3:8b", "qwen3:4b-instruct-2507-q4_K_M"]
#@markdown **KEEP_MODEL_ON_DRIVE**: مدل یک بار در Drive ذخیره شود (حدود ۹ گیگ برای 14b) تا دفعه‌های بعد دانلود نشود.
KEEP_MODEL_ON_DRIVE = False  #@param {type:"boolean"}
import os, subprocess, time
os.environ['MATRAG_LLM_PROVIDER'] = LLM_PROVIDER
os.environ['MATRAG_OLLAMA_MODEL'] = OLLAMA_MODEL
os.environ['MATRAG_OLLAMA_THINKING'] = 'false'  # faster answers from Qwen3

try:  # Gemini is optional; with a key both models can be chosen in the app
    from google.colab import userdata
    os.environ['MATRAG_GOOGLE_API_KEY'] = userdata.get('MATRAG_GOOGLE_API_KEY')
    print('✅ Gemini key loaded')
except Exception:
    print('ℹ️ No Gemini key in Colab Secrets (only needed for gemini)')

if LLM_PROVIDER == "ollama":
    os.environ['OLLAMA_MODELS'] = '/content/ollama_models'
    DRIVE_MODELS = f'{WORK}/ollama_models'
    if KEEP_MODEL_ON_DRIVE and os.path.isdir(DRIVE_MODELS):
        print('Copying saved model from Drive...')
        !mkdir -p /content/ollama_models && cp -rT "{DRIVE_MODELS}" /content/ollama_models
    if subprocess.run('which ollama', shell=True, capture_output=True).returncode != 0:
        !apt-get install -y -qq zstd > /dev/null
        !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    subprocess.Popen('ollama serve > /content/ollama.log 2>&1', shell=True)
    time.sleep(5)
    !ollama pull {OLLAMA_MODEL} 2>&1 | tail -1
    if KEEP_MODEL_ON_DRIVE:
        !mkdir -p "{DRIVE_MODELS}" && cp -rTu /content/ollama_models "{DRIVE_MODELS}" && echo "Model saved to Drive."
print('✅ Language model ready:', LLM_PROVIDER)

In [ ]:
#@title ۵. اجرای برنامه 🚀
#@markdown مجموعه‌ی پیش‌فرض (در خود برنامه هم قابل تغییر است؛ برای موضوع جدید اسم تازه تایپ کنید).
CORPUS = "hitran"  #@param ["hitran", "optical"] {allow-input: true}
#@markdown **PUBLIC_LINK**: فقط اگر پنجره‌ی برنامه پایین این سلول باز نشد روشن کنید؛ یک لینک موقت gradio.live می‌سازد
#@markdown (هر کسی که لینک را داشته باشد می‌تواند برنامه را باز کند؛ بعد از ۷۲ ساعت از کار می‌افتد).
PUBLIC_LINK = False  #@param {type:"boolean"}
import os, sys
os.environ['MATRAG_CORPUS'] = CORPUS
if '/content/RAG/src' not in sys.path:  # see cell 3
    sys.path.insert(0, '/content/RAG/src')
from matrag.webapp import launch
launch(port=7860, share=PUBLIC_LINK, inline=False, prevent_thread_lock=True, quiet=not PUBLIC_LINK)
if not PUBLIC_LINK:
    from google.colab import output
    output.serve_kernel_port_as_window(7860, anchor_text='👉 باز کردن برنامه در یک تب جدید (پیشنهاد می‌شود)')
    output.serve_kernel_port_as_iframe(7860, height=950)

---
### (اختیاری) ارزیابی

این سلول‌ها جدول‌های نتایج مقاله را می‌سازند. فایل‌ها در `matrag_data/results/eval` ذخیره می‌شوند.
سؤال‌های مرجع در `matrag_data/data/<corpus>/gold/questions.csv` قرار دارند.

In [ ]:
#@title ۶. ارزیابی بازیابی (بدون مصرف مدل زبانی)
!mkdir -p "{WORK}/data/{CORPUS}/gold" && cp -n /content/RAG/data/{CORPUS}/gold/*.csv "{WORK}/data/{CORPUS}/gold/" 2>/dev/null; true
%cd {WORK}
!matrag evaluate retrieval 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
%cd /content/RAG

In [ ]:
#@title ۷. ارزیابی پاسخ‌ها: RAG در برابر مدل بدون منبع
LIMIT = 16  #@param {type:"integer"}
%cd {WORK}
!matrag evaluate answers --limit {LIMIT} 2>&1 | grep -vE "Warning|Loading" | tee -a "{WORK}/logs/eval.log"
%cd /content/RAG